In [ ]:
study_name = None
study_id = None
model_id = None
study_dir = None

In [ ]:
import mutopia.analysis as mu
import mutopia.plot.track_plot as tr
import matplotlib.pyplot as plt
import seaborn as sns
import os

study_id = f"{study_id:02d}"
def output_path(*paths: str):
    return os.path.join("analyses", str(study_name), study_id, str(model_id), *paths)

output_dir = output_path("")
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
model = mu.load_model(f"studies/{study_name}/{study_id}/trial={model_id}.pkl")
data = mu.gt.lazy_load(f"gtensors/{study_name}.nc")

In [ ]:
ax=sns.lineplot(model.test_scores_[3:])
ax.set(xlabel="Training iteration", ylabel="Test score \u2192")
sns.despine()

In [ ]:
data = model.annot_contributions(data) # Annotate the contributions using the entire dataset!
data = mu.gt.slice_regions(data, "chr2")
data = model.annot_component_distributions(data)
data = model.annot_marginal_prediction(data)
data = model.annot_components(data)
#data = model.annot_SHAP_values(data, 0, threads=5)
data = mu.gt.annot_empirical_marginal(data)

In [ ]:
mu.gt.write_dataset(data, output_path("annotated_data.nc"), write_samples=False)

In [ ]:
mu.pl.plot_signature_panel(data)

In [ ]:
contribs = data["contributions"]
data["relative_contributions"] = contribs.squeeze()/contribs.squeeze().sum(dim="component")
_, ax = plt.subplots(figsize=(4,6))
sns.boxenplot(
    data["relative_contributions"].to_pandas().melt(ignore_index=False),
    x="value",
    y="component",
    alpha=0.5,
    ax = ax,
)
ax.set(xlim=(0,0.5), xlabel="Fraction of sample", ylabel="Mutational process")
sns.despine()

In [ ]:
g = sns.clustermap(data["relative_contributions"].to_pandas().T, cbar_pos=None, figsize=(8,6), cmap="viridis", vmax=0.35, dendrogram_ratio=(0.05,0.2))
g.ax_heatmap.set_xlabel("Sample")
g.ax_heatmap.set_xticklabels([])
None

In [ ]:
view = tr.make_view(data, "chr2:1000000-100000000")

In [ ]:
prediction_config = lambda view, scalebar_bp=1_000_000 : (
    tr.scale_bar(scalebar_bp, scale="mb" if scalebar_bp >= 1_000_000 else "kb"),
    tr.ideogram("annotations/hg38.cytoband.bed.gz"),
    tr.tracks.plot_marginal_observed_vs_expected(
        view,
        pred_smooth=7,
        smooth=7,
        height=1.25,
    ),
)

tr.plot_view(prediction_config, view, width=10)
None

In [ ]:
topography = tr.TopographyTransformer().fit(data)

topo_config = lambda view, scalebar_bp=1_000_000 : (
    prediction_config(view, scalebar_bp),
    tr.plot_topography(topography, height=2),
)
tr.plot_view(topo_config, view, width=10)
None